In [1]:
import gzip
import json
import re
import requests
from tqdm.auto import tqdm

In [2]:
META_URL = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/meta_categories/meta_Electronics.jsonl.gz"
REVIEWS_URL = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Electronics.jsonl.gz"

In [3]:
def stream_jsonl_gz(url):
    response = requests.get(url, stream=True, timeout=120)
    response.raise_for_status()

    try:
        with gzip.GzipFile(fileobj=response.raw) as gz:
            for raw_line in gz:
                if not raw_line:
                    continue
                try:
                    yield json.loads(raw_line)
                except Exception:
                    continue
    finally:
        response.close()

In [4]:
meta_stream = stream_jsonl_gz(META_URL)

for i, record in enumerate(meta_stream):
    print(record)
    if i == 2:
        break

{'main_category': 'All Electronics', 'title': 'FS-1051 FATSHARK TELEPORTER V3 HEADSET', 'average_rating': 3.5, 'rating_number': 6, 'features': [], 'description': ['Teleporter V3 The “Teleporter V3” kit sets a new level of value in the FPV world with Fat Shark renowned performance and quality. The fun of FPV is experienced firsthand through the large screen FPV headset with integrated NexwaveRF receiver technology while simultaneously recording onboard HD footage with the included “PilotHD” camera. The “Teleporter V3” kit comes complete with everything you need to step into the cockpit of your FPV vehicle. We’ve included our powerful 250mW 5.8Ghz transmitter, 25 degree FOV headset (largest QVGA display available), the brand new “PilotHD” camera with live AV out and all the cables, antennas and connectors needed.'], 'price': None, 'images': [{'thumb': 'https://m.media-amazon.com/images/I/41qrX56lsYL._AC_US40_.jpg', 'large': 'https://m.media-amazon.com/images/I/41qrX56lsYL._AC_.jpg', 'var

In [5]:
def normalize_text(value):
    if value is None:
        return ""
    if isinstance(value, list):
        return " ".join(str(x) for x in value if x is not None)
    return str(value)

In [6]:
def is_wireless_headphone_product(record):
    text_parts = [
        normalize_text(record.get("title")),
        normalize_text(record.get("features")),
        normalize_text(record.get("description")),
        normalize_text(record.get("categories")),
        normalize_text(record.get("details")),
        normalize_text(record.get("store")),
    ]

    text = " ".join(text_parts).lower()

    # Must indicate wireless
    wireless_terms = [
        "wireless",
        "bluetooth",
        "true wireless",
        "tws"
    ]

    # Must be headphone-type device
    audio_terms = [
        "headphone",
        "headphones",
        "earbud",
        "earbuds"
    ]

    # Exclude unwanted products
    exclude_terms = [
        "wired",
        "speaker",
        "projector",
        "receiver",
        "transmitter",
        "microphone only",
        "handset",
        "school headset",
        "classroom headphones",
        "gaming chair",
        "adapter",
        "charger",
        "charging cable",
        "cable",
        "case",
        "cover",
        "ear tips",
        "ear tip",
        "ear pads",
        "replacement",
        "accessory",
        "holder",
        "stand"
    ]

    has_wireless = any(term in text for term in wireless_terms)
    has_audio = any(term in text for term in audio_terms)
    has_exclusion = any(term in text for term in exclude_terms)

    # Must have both wireless + headphone/earbud, and no exclusion
    return has_wireless and has_audio and not has_exclusion

In [7]:
meta_stream = stream_jsonl_gz(META_URL)

matches = []
for record in meta_stream:
    if is_wireless_headphone_product(record):
        matches.append(record.get("title"))
    if len(matches) == 10:
        break

for title in matches:
    print(title)

Jabra 100-92200002-02 Clear Bluetooth Headset - Retail Packaging - White (Discontinued by Manufacturer)
SoundBot® SB562 Stereo Bluetooth 4.1 Sports-Active Wireless Headset High-Performance Earbud Earphone w/ Intuitive Voice Prompt, 33ft Wireless Range, for Wireless Hands-Free Talking & Music Streaming
Universal 3.5mm In-Ear Stereo Headset w/ On-off & Mic for , Samsung Transform M920, Black
Wireless Earbuds Bluetooth Headphones, A8 Game Mode Bluetooth Earbuds, IPX8 Waterproof Sport Wireless Headphones Earphone, USB-C Charging/36H Playtime/Precise Touch Control/Twin&Mono Mode
KAROTTO Wireless Bone Conduction Headphones, IPX 8 Waterproof Open-Ear Bluetooth Headphones Bulit in 8G Memory with Mic
HAYLOU PurFree Bone Conduction Headphones Open-Ear Bluetooth 5.2 Sport Headphones -IP67 Waterproof Wireless Earphones for Cycling and Running - CVC Dual Microphone Noise Reduction Call
Ghostek soDrop 2 Wireless Bluetooth Headphones with Noise Reduction Amazing Bass | White & Gold
Children's Combine

In [8]:
parent_asins = set()
matched_products = 0

with open("wireless_headphones_meta.jsonl", "w", encoding="utf-8") as fout:
    for record in tqdm(stream_jsonl_gz(META_URL), desc="Scanning metadata"):
        if is_wireless_headphone_product(record):
            parent_asin = record.get("parent_asin")
            if parent_asin:
                parent_asins.add(parent_asin)
                fout.write(json.dumps(record) + "\n")
                matched_products += 1

print("Matched metadata rows:", matched_products)
print("Unique parent_asin values:", len(parent_asins))

Scanning metadata: 0it [00:00, ?it/s]

Matched metadata rows: 8511
Unique parent_asin values: 8511


In [9]:
with open("wireless_headphones_parent_asins.txt", "w", encoding="utf-8") as f:
    for asin in sorted(parent_asins):
        f.write(asin + "\n")

In [10]:
review_stream = stream_jsonl_gz(REVIEWS_URL)

for i, record in enumerate(review_stream):
    print(record)
    if i == 2:
        break

{'rating': 3.0, 'title': 'Smells like gasoline! Going back!', 'text': 'First & most offensive: they reek of gasoline so if you are sensitive/allergic to petroleum products like I am you will want to pass on these.  Second: the phone adapter is useless as-is. Mine was not drilled far enough to be able to tighten it into place for my iPhone 12 max. It just slipped & slid all over. Stupid me putting the adapter together first without picking up the binoculars to smell them bc I wasted 15 minutes trying to figure out how to put the adapter together bc it does not come with instructions!  I had to come back here to the website which was a total pain. Third: the tripod is also useless. I would not trust the iOS to hold my $1600 phone nor even a Mattel Barbie for that matter. It’s just inefficient for the job imo.  Third: in order to try to give an honest review I did don gloves & eyewear to check the binoculars out.  They seemed average except for mine seemed to be missing about 10% of the f

In [11]:
matched_reviews = 0

with open("wireless_headphones_reviews.jsonl", "w", encoding="utf-8") as fout:
    for record in tqdm(stream_jsonl_gz(REVIEWS_URL), desc="Scanning reviews"):
        parent_asin = record.get("parent_asin")
        if parent_asin in parent_asins:
            fout.write(json.dumps(record) + "\n")
            matched_reviews += 1

print("Matched reviews:", matched_reviews)

Scanning reviews: 0it [00:00, ?it/s]

Matched reviews: 314635


In [12]:
with open("wireless_headphones_reviews.jsonl", "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        record = json.loads(line)
        print(record)
        if i == 2:
            break

{'rating': 5.0, 'title': 'Great purchase!', 'text': 'Excellent product. Great customer service.', 'images': [], 'asin': 'B01M74RA05', 'parent_asin': 'B01M74RA05', 'user_id': 'AHQQ62COZ3ANN33IRXZ6Z4KF664A', 'timestamp': 1486512426000, 'helpful_vote': 0, 'verified_purchase': True}
{'rating': 3.0, 'title': 'Battery life sucks', 'text': 'Battery lasted 2 months ...sucks', 'images': [], 'asin': 'B085L34V8J', 'parent_asin': 'B085L34V8J', 'user_id': 'AESGKFNIDELMTPT7OLS67JA545JA', 'timestamp': 1603999669246, 'helpful_vote': 0, 'verified_purchase': True}
{'rating': 1.0, 'title': 'JUNK!!!', 'text': 'These are absolute junk.  Not at all the Bose standard.  They broke within 30days.  DO NOT BUY!', 'images': [], 'asin': 'B01L7PWBRG', 'parent_asin': 'B0B62FJF1J', 'user_id': 'AEAU4B3HGL46SP3ARQZZ3B7PDA4Q', 'timestamp': 1489026041000, 'helpful_vote': 0, 'verified_purchase': False}


In [13]:
import pandas as pd

sample_reviews = []

with open("wireless_headphones_reviews.jsonl", "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        sample_reviews.append(json.loads(line))
        if i == 4999:
            break

df = pd.DataFrame(sample_reviews)
df.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5.0,Great purchase!,Excellent product. Great customer service.,[],B01M74RA05,B01M74RA05,AHQQ62COZ3ANN33IRXZ6Z4KF664A,1486512426000,0,True
1,3.0,Battery life sucks,Battery lasted 2 months ...sucks,[],B085L34V8J,B085L34V8J,AESGKFNIDELMTPT7OLS67JA545JA,1603999669246,0,True
2,1.0,JUNK!!!,These are absolute junk. Not at all the Bose ...,[],B01L7PWBRG,B0B62FJF1J,AEAU4B3HGL46SP3ARQZZ3B7PDA4Q,1489026041000,0,False
3,2.0,Poor connection,"Crappy sound quality, poor fit, volume control...",[],B07CX38CQ2,B07GC8NMB8,AE427MY65PZ6FNZ2IY6HCTIIQZEQ,1565129082806,0,True
4,4.0,These earbuds work great. It has very good sou...,These earbuds work great. It has very good so...,[],B00H36UL5I,B00H36UL5I,AHATA6X6MYTC3VNBFJ3WIYVK257A,1515081112399,0,False


In [14]:
print(df.columns.tolist())

['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']


In [15]:
df = pd.DataFrame(sample_reviews)

In [16]:
df.isnull().sum()

,0
rating,0
title,0
text,0
images,0
asin,0
parent_asin,0
user_id,0
timestamp,0
helpful_vote,0
verified_purchase,0


In [17]:
df["title"] = df["title"].fillna("")
df["text"] = df["text"].fillna("")
df["helpful_vote"] = df["helpful_vote"].fillna(0)
df["verified_purchase"] = df["verified_purchase"].fillna(False)

In [18]:
df = df[(df["title"].str.strip() != "") | (df["text"].str.strip() != "")]
df.shape

(5000, 10)

In [19]:
df["review_full"] = (
    df["title"].str.strip() + " " + df["text"].str.strip()
).str.strip()

In [20]:
import pandas as pd

df["review_time"] = pd.to_datetime(df["timestamp"], unit="ms", errors="coerce")
df["review_date"] = df["review_time"].dt.date
df["review_year"] = df["review_time"].dt.year
df["review_month"] = df["review_time"].dt.to_period("M").astype(str)

In [21]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["review_clean"] = df["review_full"].apply(clean_text)

In [22]:
df["review_length"] = df["review_clean"].str.split().str.len()

df = df[df["review_length"] >= 3]
df.shape

(4968, 17)

In [23]:
def rating_to_sentiment(rating):
    if rating <= 2:
        return "negative"
    elif rating == 3:
        return "neutral"
    else:
        return "positive"

df["sentiment_label"] = df["rating"].apply(rating_to_sentiment)

In [24]:
df["helpful_vote"] = pd.to_numeric(df["helpful_vote"], errors="coerce").fillna(0)

df["importance_score"] = (
    df["helpful_vote"] * 2 +
    (5 - df["rating"]) * 3 +
    df["review_length"] * 0.01
)

In [25]:
df = df.drop_duplicates(subset=["parent_asin", "review_full"]).copy()
df.shape

(4943, 19)

In [26]:
df.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,review_full,review_time,review_date,review_year,review_month,review_clean,review_length,sentiment_label,importance_score
0,5.0,Great purchase!,Excellent product. Great customer service.,[],B01M74RA05,B01M74RA05,AHQQ62COZ3ANN33IRXZ6Z4KF664A,1486512426000,0,True,Great purchase! Excellent product. Great custo...,2017-02-08 00:07:06.000,2017-02-08,2017,2017-02,great purchase excellent product great custome...,7,positive,0.07
1,3.0,Battery life sucks,Battery lasted 2 months ...sucks,[],B085L34V8J,B085L34V8J,AESGKFNIDELMTPT7OLS67JA545JA,1603999669246,0,True,Battery life sucks Battery lasted 2 months ......,2020-10-29 19:27:49.246,2020-10-29,2020,2020-10,battery life sucks battery lasted 2 months sucks,8,neutral,6.08
2,1.0,JUNK!!!,These are absolute junk. Not at all the Bose ...,[],B01L7PWBRG,B0B62FJF1J,AEAU4B3HGL46SP3ARQZZ3B7PDA4Q,1489026041000,0,False,JUNK!!! These are absolute junk. Not at all t...,2017-03-09 02:20:41.000,2017-03-09,2017,2017-03,junk these are absolute junk not at all the bo...,18,negative,12.18
3,2.0,Poor connection,"Crappy sound quality, poor fit, volume control...",[],B07CX38CQ2,B07GC8NMB8,AE427MY65PZ6FNZ2IY6HCTIIQZEQ,1565129082806,0,True,"Poor connection Crappy sound quality, poor fit...",2019-08-06 22:04:42.806,2019-08-06,2019,2019-08,poor connection crappy sound quality poor fit ...,15,negative,9.15
4,4.0,These earbuds work great. It has very good sou...,These earbuds work great. It has very good so...,[],B00H36UL5I,B00H36UL5I,AHATA6X6MYTC3VNBFJ3WIYVK257A,1515081112399,0,False,These earbuds work great. It has very good sou...,2018-01-04 15:51:52.399,2018-01-04,2018,2018-01,these earbuds work great it has very good soun...,41,positive,3.41


In [27]:
df.columns.tolist()

['rating',
 'title',
 'text',
 'images',
 'asin',
 'parent_asin',
 'user_id',
 'timestamp',
 'helpful_vote',
 'verified_purchase',
 'review_full',
 'review_time',
 'review_date',
 'review_year',
 'review_month',
 'review_clean',
 'review_length',
 'sentiment_label',
 'importance_score']

In [28]:
df.to_csv("wireless_headphones_reviews_cleaned_full.csv", index=False)

In [29]:
from google.colab import files
files.download("wireless_headphones_reviews_cleaned_full.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>